In [1]:
import pandas as pd

In [2]:
!gdown 1kJXtUko-70rNP250KxkiW3OTE7bg1RAo

Downloading...
From: https://drive.google.com/uc?id=1kJXtUko-70rNP250KxkiW3OTE7bg1RAo
To: /content/final_dataset.csv
100% 64.7M/64.7M [00:01<00:00, 39.0MB/s]


# Priprema podataka

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.metrics import classification_report, f1_score

# ---------------------------------------------------------
# 1) Ucitavanje i priprema podataka
# ---------------------------------------------------------
df = pd.read_csv("final_dataset.csv")

# baciti redove bez teksta (nema sta trenirati)
df = df.dropna(subset=["SADRZAJ"]).reset_index(drop=True)

TOPICS = [
    "euroatlantske_integracije",
    "negiranje_genocida",
    "gradjanska_vs_konstitutivni",
    "izborna_reforma",
]

def make_label(row, topic):
    """mentioned + stance -> not_mentioned / against / neutral / for"""
    if not row[f"{topic}_mentioned"]:
        return "not_mentioned"
    stance = row[f"{topic}_stance"]
    if stance == 1:
        return "for"
    elif stance == -1:
        return "against"
    else:
        return "neutral"

for topic in TOPICS:
    df[f"{topic}_label"] = df.apply(lambda r: make_label(r, topic), axis=1)

# ---------------------------------------------------------
# 2) TF-IDF (word + char_wb n-grami) preko FeatureUnion
# ---------------------------------------------------------
def build_vectorizer():
    return FeatureUnion([
        ("word", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=2,
            max_df=0.9,
            sublinear_tf=True,
        )),
        ("char", TfidfVectorizer(
            analyzer="char_wb",
            ngram_range=(3, 5),
            min_df=2,
            max_df=0.9,
            sublinear_tf=True,
        )),
    ])

# Dio 1: TF-IDF + SGD / Logistic Regression (baseline, bez grid searcha)

Osnovni baseline: TF-IDF vektorizacija + `LogisticRegression` i `SGDClassifier`
(sa hinge i log_loss gubitkom) sa fiksnim, razumnim hiperparametrima.
Bez optimizacije - ovo je referentna tacka za Dio 2 i Dio 3.

In [5]:
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.base import clone

MODELS_PART1 = {
    "logreg": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        C=1.0,
        solver="saga",
        n_jobs=-1,
    ),
    "sgd_hinge": SGDClassifier(
        loss="hinge",
        class_weight="balanced",
        alpha=1e-5,
        max_iter=2000,
        random_state=42,
    ),
    "sgd_log": SGDClassifier(
        loss="log_loss",
        class_weight="balanced",
        alpha=1e-5,
        max_iter=2000,
        random_state=42,
    ),
}

In [5]:
# ---------------------------------------------------------
# Treniranje posebnog modela po temi (Dio 1 - baseline)
# ---------------------------------------------------------
results_part1 = {}

for topic in TOPICS:
    print("=" * 70)
    print(f"TEMA: {topic}")
    print("=" * 70)

    X = df["SADRZAJ"]
    y = df[f"{topic}_label"]

    print("Distribucija klasa:\n", y.value_counts(), "\n")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    for model_name, clf in MODELS_PART1.items():
        pipe = Pipeline([
            ("tfidf", build_vectorizer()),
            ("clf", clone(clf)),
        ])

        pipe.fit(X_train, y_train)
        y_pred = pipe.predict(X_test)

        f1_macro = f1_score(y_test, y_pred, average="macro", zero_division=0)
        print(f"--- {model_name} | macro-F1 = {f1_macro:.3f} ---")
        print(classification_report(y_test, y_pred, zero_division=0))

        results_part1[(topic, model_name)] = {
            "pipeline": pipe,
            "f1_macro": f1_macro,
        }

TEMA: euroatlantske_integracije
Distribucija klasa:
 euroatlantske_integracije_label
not_mentioned    10868
neutral           1541
for               1354
against           1243
Name: count, dtype: int64 



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


--- logreg | macro-F1 = 0.597 ---
               precision    recall  f1-score   support

      against       0.40      0.56      0.47       249
          for       0.61      0.64      0.62       271
      neutral       0.37      0.45      0.41       308
not_mentioned       0.93      0.85      0.89      2174

     accuracy                           0.77      3002
    macro avg       0.58      0.63      0.60      3002
 weighted avg       0.80      0.77      0.78      3002

--- sgd_hinge | macro-F1 = 0.597 ---
               precision    recall  f1-score   support

      against       0.52      0.45      0.48       249
          for       0.67      0.61      0.63       271
      neutral       0.46      0.30      0.36       308
not_mentioned       0.88      0.95      0.91      2174

     accuracy                           0.81      3002
    macro avg       0.63      0.57      0.60      3002
 weighted avg       0.79      0.81      0.79      3002

--- sgd_log | macro-F1 = 0.596 ---
        

KeyboardInterrupt: 

In [ ]:
import joblib
import os

os.makedirs("models", exist_ok=True)

for (topic, model_name), res in results_part1.items():
    path = f"models/{topic}__{model_name}.joblib"
    joblib.dump(res["pipeline"], path)
    print(f"Sacuvano: {path} (f1_macro={res['f1_macro']:.3f})")

In [ ]:
# ---------------------------------------------------------
# Pregled rezultata - Dio 1
# ---------------------------------------------------------
summary_part1 = pd.DataFrame([
    {"tema": t, "model": m, "macro_f1": r["f1_macro"]}
    for (t, m), r in results_part1.items()
]).sort_values(["tema", "macro_f1"], ascending=[True, False])

print(summary_part1.to_string(index=False))

# Dio 2: TF-IDF + SGD / Logistic Regression sa GridSearchCV i dodatnom optimizacijom

Ista dva klasifikatora (Logistic Regression, SGD), ali sada tunamo i
TF-IDF (ngram_range, min_df) i hiperparametre klasifikatora preko `GridSearchCV`
sa `StratifiedKFold` i `f1_macro` kao scoring metrikom.

In [1]:
# pip install sentence-transformers

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [9]:
df["TEXT_EMB"] = (
    df["naslov"].fillna("") + ". " +
    df["SADRZAJ"].fillna("").str[:2500]
)

In [10]:
emb_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-mpnet-base-v2")

X_emb = emb_model.encode(
    df["TEXT_EMB"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

In [11]:
EMB_MODELS = {
    "linear_svc": LinearSVC(
        C=1.0,
        class_weight="balanced",
        max_iter=5000,
        random_state=42
    ),

    "logreg": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        C=1.0,
        solver="saga",
        n_jobs=-1
    ),

    "sgd_hinge": SGDClassifier(
        loss="hinge",
        class_weight="balanced",
        alpha=1e-5,
        max_iter=2000,
        random_state=42
    ),
        "sgd_log": SGDClassifier(
        loss="log_loss",
        class_weight="balanced",
        alpha=1e-5,
        max_iter=2000,
        random_state=42,
    ),
}


TEMA: euroatlantske_integracije | EMBEDDINGS
--- linear_svc | macro-F1 = 0.524 ---
               precision    recall  f1-score   support

      against       0.35      0.48      0.40       249
          for       0.46      0.55      0.51       271
      neutral       0.37      0.29      0.32       308
not_mentioned       0.88      0.85      0.87      2174

     accuracy                           0.73      3002
    macro avg       0.52      0.54      0.52      3002
 weighted avg       0.75      0.73      0.74      3002

--- logreg | macro-F1 = 0.490 ---
               precision    recall  f1-score   support

      against       0.27      0.58      0.37       249
          for       0.40      0.62      0.49       271
      neutral       0.26      0.41      0.32       308
not_mentioned       0.93      0.68      0.78      2174

     accuracy                           0.63      3002
    macro avg       0.47      0.57      0.49      3002
 weighted avg       0.76      0.63      0.67      300

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


--- logreg | macro-F1 = 0.243 ---
               precision    recall  f1-score   support

      against       0.08      0.94      0.15       127
          for       0.00      0.00      0.00       239
      neutral       0.08      0.23      0.12        48
not_mentioned       0.98      0.54      0.70      2588

     accuracy                           0.51      3002
    macro avg       0.29      0.43      0.24      3002
 weighted avg       0.85      0.51      0.61      3002

--- sgd_hinge | macro-F1 = 0.413 ---
               precision    recall  f1-score   support

      against       0.18      0.55      0.27       127
          for       0.49      0.31      0.38       239
      neutral       0.07      0.25      0.10        48
not_mentioned       0.96      0.84      0.90      2588

     accuracy                           0.78      3002
    macro avg       0.42      0.49      0.41      3002
 weighted avg       0.87      0.78      0.82      3002

TEMA: gradjanska_vs_konstitutivni | EMBEDDI

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


--- logreg | macro-F1 = 0.129 ---
               precision    recall  f1-score   support

      against       0.34      0.04      0.07       415
          for       0.00      0.00      0.00       166
      neutral       0.02      1.00      0.04        48
not_mentioned       0.97      0.26      0.40      2373

     accuracy                           0.22      3002
    macro avg       0.33      0.32      0.13      3002
 weighted avg       0.82      0.22      0.33      3002

--- sgd_hinge | macro-F1 = 0.431 ---
               precision    recall  f1-score   support

      against       0.51      0.30      0.38       415
          for       0.26      0.60      0.37       166
      neutral       0.06      0.40      0.11        48
not_mentioned       0.93      0.82      0.87      2373

     accuracy                           0.73      3002
    macro avg       0.44      0.53      0.43      3002
 weighted avg       0.82      0.73      0.76      3002

TEMA: izborna_reforma | EMBEDDINGS
--- line

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


--- logreg | macro-F1 = 0.273 ---
               precision    recall  f1-score   support

      against       0.11      0.18      0.14       129
          for       0.05      0.79      0.09        48
      neutral       0.09      0.52      0.15        82
not_mentioned       0.99      0.55      0.71      2743

     accuracy                           0.54      3002
    macro avg       0.31      0.51      0.27      3002
 weighted avg       0.91      0.54      0.66      3002

--- sgd_hinge | macro-F1 = 0.475 ---
               precision    recall  f1-score   support

      against       0.25      0.41      0.31       129
          for       0.25      0.33      0.29        48
      neutral       0.37      0.35      0.36        82
not_mentioned       0.96      0.93      0.94      2743

     accuracy                           0.88      3002
    macro avg       0.46      0.51      0.48      3002
 weighted avg       0.90      0.88      0.89      3002



In [ ]:
from sklearn.base import clone
import numpy as np

results_emb_twostage = {}

for topic in TOPICS:
    print("=" * 70)
    print(f"TEMA: {topic} | EMBEDDINGS TWO-STAGE")
    print("=" * 70)

    y = df[f"{topic}_label"]

    idx_train, idx_test, y_train, y_test = train_test_split(
        np.arange(len(df)),
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    X_train = X_emb[idx_train]
    X_test = X_emb[idx_test]

    # 1) binary label: mentioned / not_mentioned
    y_train_binary = np.where(
        y_train == "not_mentioned",
        "not_mentioned",
        "mentioned"
    )

    y_test_binary = np.where(
        y_test == "not_mentioned",
        "not_mentioned",
        "mentioned"
    )

    # 2) stance data: samo mentioned primjeri
    mentioned_train_mask = y_train != "not_mentioned"

    X_train_stance = X_train[mentioned_train_mask]
    y_train_stance = y_train[mentioned_train_mask]

    print("Binary train distribucija:")
    print(pd.Series(y_train_binary).value_counts(), "\n")

    print("Stance train distribucija:")
    print(y_train_stance.value_counts(), "\n")

    for model_name, clf in EMB_MODELS.items():
        print("-" * 50)
        print(f"MODEL: {model_name}")
        print("-" * 50)

        # Model 1: mentioned vs not_mentioned
        binary_clf = clone(clf)
        binary_clf.fit(X_train, y_train_binary)

        # Model 2: for / against / neutral
        stance_clf = clone(clf)
        stance_clf.fit(X_train_stance, y_train_stance)

        # Prvo predvidi binary
        binary_pred = binary_clf.predict(X_test)

        # Sve inicijalno postavi kao not_mentioned
        final_pred = np.array(["not_mentioned"] * len(X_test), dtype=object)

        # Gdje binary kaže mentioned, koristi stance model
        mentioned_test_mask = binary_pred == "mentioned"

        if mentioned_test_mask.sum() > 0:
            stance_pred = stance_clf.predict(X_test[mentioned_test_mask])
            final_pred[mentioned_test_mask] = stance_pred

        # Evaluacija na originalne 4 klase
        f1_macro = f1_score(
            y_test,
            final_pred,
            average="macro",
            zero_division=0
        )

        print(f"--- TWO-STAGE {model_name} | macro-F1 = {f1_macro:.3f} ---")
        print(classification_report(y_test, final_pred, zero_division=0))

        results_emb_twostage[(topic, model_name)] = {
            "binary_model": binary_clf,
            "stance_model": stance_clf,
            "f1_macro": f1_macro,
        }

In [ ]:
import joblib
import os

os.makedirs("models2", exist_ok=True)

for (topic, model_name), res in results_emb_twostage.items():
    path = f"models2/{topic}__{model_name}_binary.joblib"
    joblib.dump(res["binary_pipeline"], path)
    path = f"models2/{topic}__{model_name}_stance.joblib"
    joblib.dump(res["stance_pipeline"], path)
    print(f"Sacuvano: {path} (f1_macro={res['f1_macro']:.3f})")

In [ ]:
import joblib
import os

os.makedirs("models", exist_ok=True)

for (topic, model_name), res in results_part2.items():
    path = f"models/{topic}__{model_name}.joblib"
    joblib.dump(res["pipeline"], path)
    print(f"Sacuvano: {path} (f1_macro={res['f1_macro']:.3f})")

In [ ]:
# ---------------------------------------------------------
# Pregled rezultata - Dio 2
# ---------------------------------------------------------
summary_part2 = pd.DataFrame([
    {
        "tema": t,
        "model": m,
        "cv_f1_macro": r["cv_f1_macro"],
        "test_f1_macro": r["test_f1_macro"],
    }
    for (t, m), r in results_part2.items()
]).sort_values(["tema", "test_f1_macro"], ascending=[True, False])

print(summary_part2.to_string(index=False))

# Dio 3: TF-IDF + Embeddings (paraphrase-multilingual-mpnet-base-v2, multilingual-e5-base)

Kombinujemo rijetke TF-IDF vektore sa gustim (dense) semantickim embeddinzima
iz dva multilingualna transformer modela:

- `paraphrase-multilingual-mpnet-base-v2` - 768-dimenzionalni prostor, treniran
  za semanticku slicnost recenica/paragrafa preko vise jezika.
- `intfloat/multilingual-e5-base` - takodje 768 dim / 12 slojeva, ali ocekuje
  prefiks `"query: "` ili `"passage: "` ispred teksta radi najboljih rezultata.

Za svaki embedding model gradimo `TF-IDF + embeddings` feature matricu (hstack)
i treniramo Logistic Regression po temi, pa poredimo sa TF-IDF-only baseline-om.

In [4]:
!pip install -q sentence-transformers

In [5]:
import scipy.sparse as sp
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import StandardScaler

EMBEDDING_MODELS = {
    "mpnet": {
        "model_name": "paraphrase-multilingual-mpnet-base-v2",
        "prefix": "",  # mpnet ne zahtijeva prefiks
    },
    "e5_base": {
        "model_name": "intfloat/multilingual-e5-base",
        "prefix": "query: ",  # e5 modeli ocekuju "query: " ili "passage: " prefiks
    },
}

# ucitavamo oba modela jednom i keširamo embeddinge po tekstu, da se ne
# racunaju iznova za svaku temu
embedding_cache = {}

for key, cfg in EMBEDDING_MODELS.items():
    print(f"Ucitavam model: {cfg['model_name']} ...")
    st_model = SentenceTransformer(cfg["model_name"])
    texts_with_prefix = [cfg["prefix"] + t for t in df["SADRZAJ"].tolist()]
    embeddings = st_model.encode(
        texts_with_prefix,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    embedding_cache[key] = embeddings
    print(f"{key}: embeddings shape = {embeddings.shape}\n")

Ucitavam model: paraphrase-multilingual-mpnet-base-v2 ...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/5.12k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/469 [00:00<?, ?it/s]

e5_base: embeddings shape = (15006, 768)



In [6]:
import numpy as np
import scipy.sparse as sp

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

In [ ]:
# ---------------------------------------------------------
# TF-IDF + embeddings TWO-STAGE po temi
# ---------------------------------------------------------
results_part3_twostage = {}

for topic in TOPICS:
    print("=" * 70)
    print(f"TEMA: {topic} | TF-IDF + EMBEDDINGS TWO-STAGE")
    print("=" * 70)

    y = df[f"{topic}_label"]

    # bolje koristiti pozicije, jer embedding matrica prati redoslijed df-a
    idx_train, idx_test = train_test_split(
        np.arange(len(df)),
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    X_text_train = df.iloc[idx_train]["SADRZAJ"]
    X_text_test = df.iloc[idx_test]["SADRZAJ"]

    y_train = y.iloc[idx_train].reset_index(drop=True)
    y_test = y.iloc[idx_test].reset_index(drop=True)

    # -----------------------------------------------------
    # Binary i stance labele
    # -----------------------------------------------------
    y_train_binary = np.where(
        y_train == "not_mentioned",
        "not_mentioned",
        "mentioned"
    )

    mentioned_train_mask = (y_train != "not_mentioned").to_numpy()

    print("Originalna distribucija:")
    print(y.value_counts(), "\n")

    print("Binary train distribucija:")
    print(pd.Series(y_train_binary).value_counts(), "\n")

    print("Stance train distribucija:")
    print(y_train[mentioned_train_mask].value_counts(), "\n")

    # -----------------------------------------------------
    # TF-IDF fitovan samo na trening skupu
    # -----------------------------------------------------
    tfidf = build_vectorizer()
    X_tfidf_train = tfidf.fit_transform(X_text_train)
    X_tfidf_test = tfidf.transform(X_text_test)

    # =====================================================
    # 1) TF-IDF + svaki embedding model
    # =====================================================
    for emb_key in EMBEDDING_MODELS:
        emb = embedding_cache[emb_key]

        emb_train = emb[idx_train]
        emb_test = emb[idx_test]

        # standardizacija dense embeddinga
        scaler = StandardScaler()
        emb_train_scaled = scaler.fit_transform(emb_train)
        emb_test_scaled = scaler.transform(emb_test)

        X_train_combined = sp.hstack([
            X_tfidf_train,
            sp.csr_matrix(emb_train_scaled)
        ]).tocsr()

        X_test_combined = sp.hstack([
            X_tfidf_test,
            sp.csr_matrix(emb_test_scaled)
        ]).tocsr()

        # -----------------------------
        # Model 1: mentioned/not_mentioned
        # -----------------------------
        binary_clf = LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            C=1.0,
            solver="saga",
            n_jobs=-1
        )

        binary_clf.fit(X_train_combined, y_train_binary)

        # -----------------------------
        # Model 2: against/for/neutral
        # -----------------------------
        stance_clf = LogisticRegression(
            max_iter=3000,
            class_weight="balanced",
            C=1.0,
            solver="saga",
            n_jobs=-1
        )

        stance_clf.fit(
            X_train_combined[mentioned_train_mask],
            y_train[mentioned_train_mask]
        )

        # -----------------------------
        # Two-stage predikcija
        # -----------------------------
        binary_pred = binary_clf.predict(X_test_combined)

        final_pred = np.array(
            ["not_mentioned"] * len(y_test),
            dtype=object
        )

        mentioned_test_mask = binary_pred == "mentioned"

        if mentioned_test_mask.sum() > 0:
            stance_pred = stance_clf.predict(
                X_test_combined[mentioned_test_mask]
            )
            final_pred[mentioned_test_mask] = stance_pred

        f1_macro = f1_score(
            y_test,
            final_pred,
            average="macro",
            zero_division=0
        )

        model_label = f"two_stage_tfidf+{emb_key}"

        print(f"--- {model_label} | macro-F1 = {f1_macro:.3f} ---")
        print(classification_report(y_test, final_pred, zero_division=0))

        results_part3_twostage[(topic, model_label)] = {
            "tfidf": tfidf,
            "scaler": scaler,
            "binary_model": binary_clf,
            "stance_model": stance_clf,
            "f1_macro": f1_macro,
        }

    # =====================================================
    # 2) Čisti TF-IDF TWO-STAGE baseline
    # =====================================================
    binary_clf_tfidf = LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        C=1.0,
        solver="saga",
        n_jobs=-1
    )

    stance_clf_tfidf = LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        C=1.0,
        solver="saga",
        n_jobs=-1
    )

    binary_clf_tfidf.fit(X_tfidf_train, y_train_binary)

    stance_clf_tfidf.fit(
        X_tfidf_train[mentioned_train_mask],
        y_train[mentioned_train_mask]
    )

    binary_pred = binary_clf_tfidf.predict(X_tfidf_test)

    final_pred = np.array(
        ["not_mentioned"] * len(y_test),
        dtype=object
    )

    mentioned_test_mask = binary_pred == "mentioned"

    if mentioned_test_mask.sum() > 0:
        stance_pred = stance_clf_tfidf.predict(
            X_tfidf_test[mentioned_test_mask]
        )
        final_pred[mentioned_test_mask] = stance_pred

    f1_macro_tfidf_only = f1_score(
        y_test,
        final_pred,
        average="macro",
        zero_division=0
    )

    print(f"--- two_stage_tfidf_only | macro-F1 = {f1_macro_tfidf_only:.3f} ---")
    print(classification_report(y_test, final_pred, zero_division=0))

    results_part3_twostage[(topic, "two_stage_tfidf_only")] = {
        "tfidf": tfidf,
        "binary_model": binary_clf_tfidf,
        "stance_model": stance_clf_tfidf,
        "f1_macro": f1_macro_tfidf_only,
    }

TEMA: euroatlantske_integracije | TF-IDF + EMBEDDINGS TWO-STAGE
Originalna distribucija:
euroatlantske_integracije_label
not_mentioned    10868
neutral           1541
for               1354
against           1243
Name: count, dtype: int64 

Binary train distribucija:
not_mentioned    8694
mentioned        3310
Name: count, dtype: int64 

Stance train distribucija:
euroatlantske_integracije_label
neutral    1233
for        1083
against     994
Name: count, dtype: int64 

--- two_stage_tfidf+mpnet | macro-F1 = 0.538 ---
               precision    recall  f1-score   support

      against       0.36      0.47      0.41       249
          for       0.49      0.54      0.51       271
      neutral       0.33      0.37      0.35       308
not_mentioned       0.91      0.85      0.88      2174

     accuracy                           0.74      3002
    macro avg       0.52      0.56      0.54      3002
 weighted avg       0.77      0.74      0.75      3002

--- two_stage_tfidf+e5_base | mac

In [ ]:
summary_part3_twostage = []

for (topic, model_label), info in results_part3_twostage.items():
    summary_part3_twostage.append({
        "topic": topic,
        "model": model_label,
        "macro_f1": info["f1_macro"]
    })

summary_part3_twostage = pd.DataFrame(summary_part3_twostage)

summary_part3_twostage.sort_values(
    ["topic", "macro_f1"],
    ascending=[True, False]
)

In [ ]:
# ---------------------------------------------------------
# Pregled rezultata - Dio 3
# ---------------------------------------------------------
summary_part3 = pd.DataFrame([
    {"tema": t, "model": m, "macro_f1": r["f1_macro"]}
    for (t, m), r in results_part3.items()
]).sort_values(["tema", "macro_f1"], ascending=[True, False])

print(summary_part3.to_string(index=False))

# Dio 4: BERTic

In [ ]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

MODEL_NAME = "classla/bcms-bertic"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TOPICS = [
    "euroatlantske_integracije",
    "negiranje_genocida",
    "gradjanska_vs_konstitutivni",
    "izborna_reforma",
]

# ---------------------------------------------------------
# 1) Priprema podataka
# ---------------------------------------------------------
df = pd.read_csv("final_dataset_10000.csv").dropna(subset=["SADRZAJ"]).reset_index(drop=True)

def make_label(row, topic):
    if not row[f"{topic}_mentioned"]:
        return "not_mentioned"
    stance = row[f"{topic}_stance"]
    if stance == 1:
        return "for"
    elif stance == -1:
        return "against"
    else:
        return "neutral"

for topic in TOPICS:
    df[f"{topic}_label"] = df.apply(lambda r: make_label(r, topic), axis=1)

LABELS_4CLASS = ["not_mentioned", "neutral", "against", "for"]
LABELS_STAGE_A = ["not_mentioned", "mentioned"]
LABELS_STAGE_B = ["neutral", "against", "for"]

# ---------------------------------------------------------
# 2) Torch Dataset wrapper
# ---------------------------------------------------------
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, label2id, max_length=256):
        self.encodings = tokenizer(
            list(texts), truncation=True, max_length=max_length
        )
        self.labels = [label2id[l] for l in labels]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

# ---------------------------------------------------------
# 3) Weighted loss zbog neuravnoteženih klasa
# ---------------------------------------------------------
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro", zero_division=0)}

# ---------------------------------------------------------
# 4) Generička funkcija za trening jednog klasifikatora
# ---------------------------------------------------------
def train_bertic_classifier(texts, labels, label_list, output_dir, epochs=5):
    label2id = {l: i for i, l in enumerate(label_list)}
    id2label = {i: l for l, i in label2id.items()}

    X_train, X_test, y_train, y_test = train_test_split(
        texts, labels, test_size=0.2, random_state=42, stratify=labels
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_ds = TextDataset(X_train, y_train, tokenizer, label2id)
    test_ds = TextDataset(X_test, y_test, tokenizer, label2id)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        num_labels=len(label_list),
        id2label=id2label,
        label2id=label2id,
    ).to(DEVICE)

    class_weights = torch.tensor(
        compute_class_weight(
            "balanced",
            classes=np.arange(len(label_list)),
            y=[label2id[l] for l in y_train],
        ),
        dtype=torch.float,
    )

    args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=epochs,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        learning_rate=2e-5,
        weight_decay=0.01,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        logging_steps=10,
        report_to="none",
    )

    trainer = WeightedTrainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        data_collator=DataCollatorWithPadding(tokenizer),
        compute_metrics=compute_metrics,
        class_weights=class_weights,
    )

    trainer.train()

    preds = np.argmax(trainer.predict(test_ds).predictions, axis=-1)
    y_true = [label2id[l] for l in y_test]
    print(classification_report(
        y_true, preds, target_names=label_list, zero_division=0
    ))

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    return trainer

# ---------------------------------------------------------
# 5a) SINGLE-STAGE: 4 odvojena BERTić modela (jedan po temi)
# ---------------------------------------------------------
single_stage_trainers = {}
for topic in TOPICS:
    print("=" * 70, f"\nBERTić SINGLE-STAGE: {topic}\n", "=" * 70)
    single_stage_trainers[topic] = train_bertic_classifier(
        texts=df["SADRZAJ"],
        labels=df[f"{topic}_label"],
        label_list=LABELS_4CLASS,
        output_dir=f"./bertic_{topic}",
    )

# ---------------------------------------------------------
# 5b) TWO-STAGE (opciono, robusnije za rijetke klase):
#     Model A: not_mentioned vs mentioned
#     Model B: for / against / neutral (samo na mentioned=True)
# ---------------------------------------------------------
two_stage_trainers = {}
for topic in TOPICS:
    print("=" * 70, f"\nBERTić TWO-STAGE — Model A ({topic})\n", "=" * 70)
    stage_a_labels = df[f"{topic}_mentioned"].map(
        {False: "not_mentioned", True: "mentioned"}
    )
    trainer_a = train_bertic_classifier(
        texts=df["SADRZAJ"],
        labels=stage_a_labels,
        label_list=LABELS_STAGE_A,
        output_dir=f"./bertic_{topic}_stageA",
    )

    mentioned_df = df[df[f"{topic}_mentioned"] == True].reset_index(drop=True)
    print("=" * 70, f"\nBERTić TWO-STAGE — Model B ({topic}), n={len(mentioned_df)}\n", "=" * 70)
    trainer_b = None
    if len(mentioned_df) >= 20:  # premalo primjera inače nema smisla trenirati
        trainer_b = train_bertic_classifier(
            texts=mentioned_df["SADRZAJ"],
            labels=mentioned_df[f"{topic}_label"],
            label_list=LABELS_STAGE_B,
            output_dir=f"./bertic_{topic}_stageB",
        )
    else:
        print(f"Preskačem Model B — samo {len(mentioned_df)} 'mentioned' primjera.")

    two_stage_trainers[topic] = {"stage_a": trainer_a, "stage_b": trainer_b}